### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are pairing at:

1. A schema, including the name of the tool, a description and/or argument definations (often a JSON schema)
2. A function or coroutinue to execute.

In [3]:
import os
import langchain

from dotenv import load_dotenv
load_dotenv()

True

In [4]:

from langchain.chat_models import init_chat_model
# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") # This line is optional

model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("Who are you?")
print(response.content)

<think>
Okay, the user is asking "Who are you?" which is a common way to start a conversation. First, I need to introduce myself clearly. I should mention my name, Qwen3, as the latest large language model developed by Alibaba. I should highlight my capabilities, like answering questions, creating text, logical reasoning, coding, etc. Also, it's important to note my multilingual support.

Next, the user might want to know my specific functions or how they can interact with me. I should keep the response friendly and open-ended, encouraging them to ask more questions or request specific tasks. I should avoid using technical jargon to keep it accessible.

I need to make sure the response is concise but informative. Let me structure it step by step: start with my name and purpose, list my key features, mention multilingual support, and invite the user to ask for help. Also, check for any recent updates or features that should be highlighted.

Wait, the user might also be interested in kno

In [8]:
## Tools

from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at a location """
    return f"It's sunny in {location} " # Here you can also use API


model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What is the weather in Bangladesh")
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Bangladesh. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since Bangladesh is a country, I should use that as the location. I need to make sure the function is called correctly with the parameter "Bangladesh". No other tools are provided, so this is the right one to use.\n', 'tool_calls': [{'id': 'kqt4mhjg8', 'function': {'arguments': '{"location":"Bangladesh"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 152, 'total_tokens': 255, 'completion_time': 0.178742555, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.006372538, 'prompt_tokens_details': None, 'queue_time': 0.159207852, 'total_time': 0.185115093}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_call

In [9]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangladesh'},
  'id': 'kqt4mhjg8',
  'type': 'tool_call'}]

#### Tool Execution Loop

In [10]:
# Step1 : Model generates tool calls
messages = [{"role":"user","content":"Weather of Dhaka and New Delhi"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The weather in Dhaka is sunny.  
The weather in New Delhi is also sunny.


In [11]:
messages

[{'role': 'user', 'content': 'Weather of Dhaka and New Delhi'},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Dhaka and New Delhi. Let me check the tools available. There's a function called get_weather that takes a location parameter. Since they want the weather for two cities, I need to call this function twice, once for each location. I'll make sure to structure each tool call correctly with the city name as the argument. Let me format the JSON for each call properly.\n", 'tool_calls': [{'id': '1bx8bs1fk', 'function': {'arguments': '{"location":"Dhaka"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'jc43fz8jb', 'function': {'arguments': '{"location":"New Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 153, 'total_tokens': 286, 'completion_time': 0.2038856, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.0

In [12]:
model_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C62C7B42D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C62B663890>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])